# Predictors and leakage checks

Builds the 83 predictors from data on or before the forecast day, and checks with 300 random cut-offs (plus a deliberately leaky negative control) that no predictor uses future data.

**Thesis:** Section 3.5 · Table 3.2 · Appendix C.5  
**Input:** predictor_table_1982_2025.csv  
**Output:** features_det_t+7.csv, features_raw_t+7.csv, feature_groups.json

Runs on Google Colab with the folder layout described in the repository README (`/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/...`). The printed outputs below are from the original run used in the thesis.

P3-1 — Setup, fixed references, feature builder

In [1]:
from google.colab import drive; drive.mount('/content/drive')
import os, json, numpy as np, pandas as pd

PROC = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/processed'
df = pd.read_csv(f'{PROC}/predictor_table_1982_2025.csv', index_col=0, parse_dates=True)
H  = 7                                                       # forecast lead (days)

def doy366(idx):
    d = idx.dayofyear.values.copy()
    d[(~idx.is_leap_year) & (idx.month > 2)] += 1
    return d

def hobday_clim(s, base=(1982, 2021), win=5, smooth=31):
    v, d = s.values, doy366(s.index)
    inb = (s.index.year >= base[0]) & (s.index.year <= base[1])
    clim = np.full(367, np.nan)
    for k in range(1, 367):
        if k == 60: continue
        idx = np.where((d == k) & inb)[0]
        pool = np.unique(np.clip((idx[:, None] + np.arange(-win, win + 1)).ravel(), 0, len(v) - 1))
        clim[k] = np.nanmean(v[pool[inb[pool]]])
    clim[60] = (clim[59] + clim[61]) / 2
    c = clim[1:]
    c = np.convolve(np.r_[c[-(smooth // 2):], c, c[:smooth // 2]], np.ones(smooth) / smooth, 'valid')
    return pd.Series(c[d - 1], index=s.index)

def doy_clim(s):                                             # fixed 1982-2021 day-of-year mean, 31-day smoothed
    base = s[s.index.year <= 2021]
    c = base.groupby(doy366(base.index)).mean()
    c = pd.Series(np.convolve(np.r_[c.values[-15:], c.values, c.values[:15]], np.ones(31) / 31, 'valid'), index=c.index)
    return pd.Series(c.reindex(doy366(s.index)).values, index=s.index)

# fixed reference series: calendar functions of the 1982-2021 baseline, identical for every fold
ERA = ['t2m', 'wspd', 'u10', 'v10', 'swnet', 'lhf', 'shf', 'qnet']
REF = pd.DataFrame(index=df.index)
REF['clim_raw'], REF['thr_raw'] = df.clim_mean, df.thresh90
REF['clim_det'] = hobday_clim(df.sst_detrended)
REF['thr_det']  = df.thresh90_detrended
for v in ERA: REF[f'clim_{v}'] = doy_clim(df[v])

def run_length(b):                                           # consecutive True days ending at t
    g = (~b).cumsum()
    return b.astype(int).groupby(g).cumsum()

def build_features(d, ref, kind):
    """kind = 'det' or 'raw'. Only rolling / shift / diff on past rows, plus fixed references."""
    sst  = d.sst_detrended if kind == 'det' else d.sst
    clim, thr = ref[f'clim_{kind}'].loc[d.index], ref[f'thr_{kind}'].loc[d.index]
    a, dist = sst - clim, sst - thr
    X = pd.DataFrame(index=d.index)
    # SST group
    X['sst_anom'], X['sst_dist_thr'] = a, dist
    for k in (1, 3, 7):        X[f'sst_anom_lag{k}'] = a.shift(k)
    for w in (3, 7, 14, 30):   X[f'sst_anom_rm{w}'] = a.rolling(w).mean()
    X['sst_anom_sd7'], X['sst_anom_max7'] = a.rolling(7).std(), a.rolling(7).max()
    for k in (1, 3, 7):        X[f'sst_anom_chg{k}'] = a - a.shift(k)
    exc = dist > 0
    X['exceed_now'] = exc.astype(int)
    X['exceed_run'] = run_length(exc)
    X['mhw_causal'] = (X.exceed_run >= 5).astype(int)        # MHW state knowable on day t
    # ERA5 group (anomalies from the fixed 1982-2021 climatology)
    for v in ERA:
        x = d[v] - ref[f'clim_{v}'].loc[d.index]
        X[f'{v}_anom'] = x
        for w in (3, 7, 14, 30): X[f'{v}_rm{w}'] = x.rolling(w).mean()
        for k in (3, 7):         X[f'{v}_chg{k}'] = x - x.shift(k)
    # climate-index group
    for v in ('nino34', 'dmi', 'iobw_dt', 'rmm1', 'rmm2'): X[v] = d[v]
    X['dmi_rm30'], X['iobw_dt_rm30'] = d.dmi.rolling(30).mean(), d.iobw_dt.rolling(30).mean()
    X['rmm_amp'] = np.hypot(d.rmm1, d.rmm2)
    # calendar group
    doy = d.index.dayofyear
    X['doy_sin'], X['doy_cos'] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    return X

GROUPS = lambda cols: {
    'sst':      [c for c in cols if c.startswith(('sst_', 'exceed_', 'mhw_causal'))],
    'era5':     [c for c in cols if c.split('_')[0] in ERA],
    'indices':  [c for c in cols if c.startswith(('nino34', 'dmi', 'iobw', 'rmm'))],
    'calendar': ['doy_sin', 'doy_cos']}

Mounted at /content/drive


P3-2 — Targets and feature tables

In [2]:
TGT = {'det': 'mhw_detrended', 'raw': 'mhw_raw'}
tables = {}
for kind, lab in TGT.items():
    X = build_features(df, REF, kind)
    y = df[lab].shift(-H).rename('y')                        # label on day t+7
    T = pd.concat([X, y], axis=1)
    T['year'] = T.index.year                                 # year of forecast day t (fold assignment)
    T['target_year'] = (T.index + pd.Timedelta(days=H)).year # year of the target (purging in Phase 4)
    T = T.dropna()
    T['y'] = T.y.astype(int)
    tables[kind] = T
    T.to_csv(f'{PROC}/features_{kind}_t+{H}.csv')
    print(f'{kind}: {T.shape[0]} rows x {X.shape[1]} features | {T.index[0].date()} -> {T.index[-1].date()} '
          f'| MHW share {T.y.mean():.1%}')
groups = GROUPS(list(X.columns))
json.dump(groups, open(f'{PROC}/feature_groups.json', 'w'), indent=1)
print({g: len(c) for g, c in groups.items()}, '| total', sum(len(c) for c in groups.values()))
assert sum(len(c) for c in groups.values()) == X.shape[1], 'a feature is missing from the groups'

det: 16035 rows x 83 features | 1982-01-30 -> 2025-12-24 | MHW share 7.4%
raw: 16035 rows x 83 features | 1982-01-30 -> 2025-12-24 | MHW share 10.3%
{'sst': 17, 'era5': 56, 'indices': 8, 'calendar': 2} | total 83


P3-3 — Leakage checks (the gate)

In [3]:
rng  = np.random.default_rng(42)
pick = np.sort(rng.choice(np.arange(40, len(df) - 1), 300, replace=False))
res = {}
for kind in TGT:
    full = build_features(df, REF, kind)
    bad = set()
    for i in pick:                                           # rebuild using ONLY data up to day t
        part = build_features(df.iloc[:i + 1], REF, kind).iloc[-1]
        diff = (part - full.iloc[i]).abs()
        bad |= set(diff[(diff > 1e-9) | (part.isna() != full.iloc[i].isna())].index)
    res[f'{kind}: features use only days <= t (300 random cut-offs)'] = len(bad) == 0
    if bad: print(kind, 'FEATURES THAT SEE THE FUTURE:', sorted(bad))
    T = tables[kind]
    lab = df[TGT[kind]]
    res[f'{kind}: y(t) = label(t+7) on every row'] = bool(
        (T.y.values == lab.reindex(T.index + pd.Timedelta(days=H)).values).all())
    res[f'{kind}: no label column among features'] = not any(c in T.columns for c in
        ['mhw_raw', 'mhw_detrended', 'sst', 'sst_detrended', 'thresh90', 'clim_mean', 'ssta'])

# negative control: a deliberately leaky feature (tomorrow's SST anomaly) must be caught
def leaky(d, ref, kind):
    X = build_features(d, ref, kind); X['leak_test'] = X.sst_anom.shift(-1); return X
caught = False
for i in pick[:50]:
    caught |= bool(pd.isna(leaky(df.iloc[:i + 1], REF, 'det').iloc[-1]['leak_test']))
res['negative control: test catches a deliberately leaky feature'] = caught

# detrending: the label's trend is fitted on all years; how much does any one year move it?
t = (df.index - df.index[0]).days.values / 3652.5
p_full = np.polyfit(t, df.sst.values, 1)
shift = 0.0
for y in df.index.year.unique():
    m = df.index.year == y
    p = np.polyfit(t[~m], df.sst.values[~m], 1)
    shift = max(shift, np.abs(np.polyval(p, t[m]) - np.polyval(p_full, t[m])).max())
print(f'trend {p_full[0]:.3f} degC/decade; leaving out any one year shifts that year\'s detrended SST by at most {shift:.3f} degC')
res['detrending: one year shifts its detrended SST by < 0.05 degC'] = shift < 0.05

print()
for k, v in res.items(): print('PASS' if v else 'FAIL', k)
print('\nGATE PASSED' if all(res.values()) else '\nGATE FAILED - send me the output')

trend 0.211 degC/decade; leaving out any one year shifts that year's detrended SST by at most 0.031 degC

PASS det: features use only days <= t (300 random cut-offs)
PASS det: y(t) = label(t+7) on every row
PASS det: no label column among features
PASS raw: features use only days <= t (300 random cut-offs)
PASS raw: y(t) = label(t+7) on every row
PASS raw: no label column among features
PASS negative control: test catches a deliberately leaky feature
PASS detrending: one year shifts its detrended SST by < 0.05 degC

GATE PASSED
